In [5]:
from dotenv import load_dotenv
load_dotenv()

from langchain_groq import ChatGroq
from langchain_core.messages import HumanMessage, SystemMessage, trim_messages
from langgraph.graph import StateGraph, MessagesState, START
from langgraph.checkpoint.memory import InMemorySaver

model = ChatGroq(model="openai/gpt-oss-120b")

# Keep only the recent part of the chat so it fits the model's context
trimmer = trim_messages(max_tokens=2000, strategy="last",
                        token_counter=model, include_system=True, start_on="human")

def call_model(state: MessagesState):
    system = SystemMessage(content="You are a helpful assistant.")
    messages = trimmer.invoke([system] + state["messages"])
    return {"messages": [model.invoke(messages)]}

workflow = StateGraph(MessagesState)
workflow.add_node("model", call_model)
workflow.add_edge(START, "model")
app = workflow.compile(checkpointer=InMemorySaver())
config = {"configurable": {"thread_id": "chat1"}}
result =app.invoke({"messages": [HumanMessage("Hi, I'm Mohammad, a frontend developer")]}, config)
print(result["messages"][-1].content)
print('------------------------------------------------------')
result = app.invoke({"messages": [HumanMessage("What's my name and job?")]}, config)
print(result["messages"][-1].content)

Hey Mohammad! 👋 Great to meet a fellow frontend developer. How’s everything going with your projects? Anything specific you’d like to chat about—new frameworks, UI challenges, performance tweaks, or just sharing what you’re working on? I’m here to help!
------------------------------------------------------
Your name is **Mohammad**, and you work as a **frontend developer**.
